## Unbiased Dynamic Volatility Strategy
- This implements a low-volatility long-short strategy. It utilizes the realized volatility of a fixed stock pool over the past N trading days, going long on the lowest-volatility stock and shorting the highest-volatility stock at each rebalancing, allocating remaining funds to the cash proxy BIL.

- It uses a custom VolatilityLongShort class in AllocationStrategy, reading only previously visible historical data through prices_history to avoid using future information for stock selection.

- Two universes are tested: a diversified universe (stocks across multiple sectors) and a tech-only universe, to examine whether universe composition affects strategy performance.

#### Benchmarks: 
Three buy-and-hold benchmarks are used to examine whether the strategy truly outperforms simple holding.
1. 100% SPY
2. 100% QQQ
3. a balanced portfolio of 70% QQQ + 20% GLD + 10% BIL

## Data Loading and Preparation

In [13]:
import pandas as pd
import numpy as np
from typing import Any
from tiportfolio.allocation import AllocationStrategy
from tiportfolio.allocation import FixRatio
from tiportfolio.calendar import Schedule
from tiportfolio.engine import ScheduleBasedEngine
from tiportfolio.report import compare_strategies

In [14]:
# Prepare Universe stocks for dynamic stock selection
UNIVERSE_1 = ['AAPL', 'MSFT', 'NVDA', 'INTC', 'AMD', 'KO', 'PEP', 'PG', 'JNJ', 'WMT', 'META', 'AMZN', 'GOOGL', 
'TSLA', 'NFLX', 'JPM', 'C', 'PGR', 'XOM', 'OXY', 'DIS', 'CCL', 'CMCSA', 'HD', 'WHR', 'LEN', 'TGT', 'MCD', 
'SBUX', 'WM', 'BA', 'CAT', 'LLY', 'PFE', 'CVX', 'COST']
UNIVERSE_2 = ['AAPL', 'MSFT', 'NVDA', 'INTC', 'AMD', 'META', 'TSLA', 'GOOGL']
CASH = 'BIL'
START = '2018-01-01'
END   = '2024-12-31'

INITIAL_VALUE = 100000
LOOKBACK = 126
TOLERANCE = 0.05
FEE = 0.0035

all_symbols = UNIVERSE + [CASH]

In [15]:
class VolatilityLongShort(AllocationStrategy):

    def __init__(self, universe, cash_symbol, lookback_days=126):
        self.universe = universe
        self.cash_symbol = cash_symbol
        self.lookback_days = lookback_days

    def get_symbols(self):
        return self.universe + [self.cash_symbol]

    def get_target_weights(self, date, total_equity, positions_dollars, prices_row, **context):
        prices_history = context.get("prices_history")

        if prices_history is None or len(prices_history) < self.lookback_days + 1:
            return {self.cash_symbol: 1.0}

        hist = prices_history[self.universe].tail(self.lookback_days + 1)

        daily_returns = hist.pct_change().dropna()
        vols = daily_returns.std() * np.sqrt(252)
        vols = vols.dropna().sort_values(ascending=True)

        long_stock  = vols.index[0]   # Long: Lowest volatility
        short_stock = vols.index[-1]  # Short: Highest volatility

        return {
            long_stock:       0.5,
            short_stock:     -0.5,
            self.cash_symbol: 1.0,
        }

In [16]:
# Volatility Strategy
strat_volatility_1 = VolatilityLongShort(
    universe=UNIVERSE_1,
    cash_symbol=CASH,
    lookback_days=LOOKBACK
)
engine_volatility_1 = ScheduleBasedEngine(
    allocation=strat_volatility_1,
    rebalance=Schedule('month_end'),
    initial_value=INITIAL_VALUE,
    fee_per_share=0.0035
)
res_volatility_1 = engine_volatility_1.run(symbols=UNIVERSE_1 + [CASH], start=START, end=END)

strat_volatility_2 = VolatilityLongShort(
    universe=UNIVERSE_2,
    cash_symbol=CASH,
    lookback_days=LOOKBACK
)
engine_volatility_2 = ScheduleBasedEngine(
    allocation=strat_volatility_2,
    rebalance=Schedule('month_end'),
    initial_value=INITIAL_VALUE,
    fee_per_share=0.0035
)
res_volatility_2 = engine_volatility_2.run(symbols=UNIVERSE_2 + [CASH], start=START, end=END)

# Benchmark 1: 100% SPY (Buy & Hold)
strat_spy = FixRatio(weights={'SPY': 1.0})
engine_spy = ScheduleBasedEngine(
    allocation=strat_spy,
    rebalance=Schedule('never'),
    initial_value=INITIAL_VALUE,
    fee_per_share=0
)
res_spy = engine_spy.run(symbols=['SPY'], start=START, end=END)

# Benchmark 2: 100% QQQ (Buy & Hold) 
strat_qqq = FixRatio(weights={'QQQ': 1.0})
engine_qqq = ScheduleBasedEngine(
    allocation=strat_qqq,
    rebalance=Schedule('never'),
    initial_value=INITIAL_VALUE,
    fee_per_share=0
)
res_qqq = engine_qqq.run(symbols=['QQQ'], start=START, end=END)

# Benchmark 3: 70% QQQ + 20% GLD + 10% BIL (Buy & Hold) 
strat_com = FixRatio(weights={'QQQ': 0.7, 'GLD': 0.2, 'BIL': 0.1})
engine_com = ScheduleBasedEngine(
    allocation=strat_com,
    rebalance=Schedule('never'),
    initial_value=INITIAL_VALUE,
    fee_per_share=0
)
res_com = engine_com.run(symbols=['QQQ', 'GLD', 'BIL'], start=START, end=END)

# Comparing 4 results
compare_strategies(res_volatility_1, res_volatility_2, res_spy, res_qqq, res_com,
                   names=['Volatility L/S Diverse', 'Volatility L/S Tech','SPY Hold', 'QQQ Hold', 'QQQ+GLD+BIL'])

Loading bar data...
Loaded bar data: 0:00:05 

Loading bar data...
Loaded bar data: 0:00:02 

Loading bar data...
Loaded bar data: 0:00:01 

Loading bar data...
Loaded bar data: 0:00:01 

Loading bar data...
Loaded bar data: 0:00:02 



,Volatility L/S Diverse,Volatility L/S Tech,SPY Hold,QQQ Hold,QQQ+GLD+BIL,better
metric,,,,,,
sharpe_ratio,-0.859347,-1.146904,0.555964,0.685937,0.691072,QQQ+GLD+BIL
sortino_ratio,-1.054541,-1.393764,0.677882,0.890879,0.902005,QQQ+GLD+BIL
mar_ratio,-0.324034,-0.312443,0.403165,0.549448,0.539188,QQQ Hold
cagr,-0.299932,-0.286057,0.136180,0.192347,0.164253,QQQ Hold
max_drawdown,0.925620,0.915550,0.337778,0.350073,0.304630,QQQ+GLD+BIL


### Result:
The volatility strategy failed, particularly evident in the tech stock universe. This is because these highly volatile stocks were also the strongest performers during this period, leading to significant losses when shorting them.